# 03 — Statistical Analysis

**Research Question:** Which SMART attributes are most associated with drive failure? — using the labelled snapshot dataset built.

This notebook works entirely in pandas/scipy/scikit-learn, since the snapshot dataset is already collapsed to one row per drive
(thousands of rows), which should fit into memory without any problems.


In [1]:
import sys
sys.path.append("../src")
import pandas as pd
import stats_utils as su

snapshot = pd.read_parquet("../data/processed/snapshot_dataset.parquet")
feature_cols = [c for c in snapshot.columns if c.startswith("smart_") and c.endswith("_raw")]

print(snapshot.shape)
print("Features:", feature_cols)
snapshot.head()

(249058, 18)
Features: ['smart_1_raw', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_7_raw', 'smart_9_raw', 'smart_10_raw', 'smart_12_raw', 'smart_192_raw', 'smart_193_raw', 'smart_194_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw']


,serial_number,model,snapshot_date,smart_1_raw,smart_3_raw,smart_4_raw,smart_5_raw,smart_7_raw,smart_9_raw,smart_10_raw,smart_12_raw,smart_192_raw,smart_193_raw,smart_194_raw,smart_197_raw,smart_198_raw,smart_199_raw,label
0,AAHKSV6H,HGST HUH721212ALN604,2025-03-31,0.0,406.0,24.0,0.0,0.0,49423.0,0.0,24.0,1824.0,1824.0,30.0,0.0,0.0,0.0,0
1,AAHKXKRH,HGST HUH721212ALN604,2025-03-31,0.0,408.0,10.0,0.0,0.0,49446.0,0.0,10.0,1786.0,1786.0,30.0,0.0,0.0,0.0,0
2,AAHLBTRH,HGST HUH721212ALN604,2025-03-31,0.0,411.0,14.0,0.0,0.0,49436.0,0.0,14.0,1990.0,1990.0,38.0,0.0,0.0,0.0,0
3,AAHLBTZH,HGST HUH721212ALN604,2025-03-31,0.0,414.0,11.0,35.0,0.0,49433.0,0.0,9.0,1825.0,1825.0,32.0,8.0,1.0,0.0,0
4,AAHLXJTH,HGST HUH721212ALN604,2025-03-31,0.0,0.0,8.0,0.0,0.0,49445.0,0.0,8.0,2005.0,2005.0,37.0,0.0,0.0,0.0,0


## 1. Univariate Comparison: Mann-Whitney U per attribute

Why Mann-Whitney rather than a t-test: SMART attributes are typically right-skewed
(most drives near zero, a long tail for degrading ones), and the two groups are
extremely unequal in size. Mann-Whitney doesn't assume normality or equal variance, 
so it's the more compliant choice in this case.

Also reporting the **rank-biserial effect size** alongside the p-value. With an
imbalanced sample, a tiny real difference can still produce a very small p-value, 
so significance alone may not mean that attributes matter in practice.
Effect size is the number to lean on when judging whether a result is meaningful, 
not just statistically detectable.

In [2]:
mw_results = su.mann_whitney_by_attribute(snapshot, feature_cols)
mw_results

,attribute,u_stat,p_value,effect_size_rank_biserial,healthy_median,failed_median
0,smart_5_raw,173176590.5,0.000000e+00,0.6059,0.0,9.0
1,smart_197_raw,176189534.0,0.000000e+00,0.6339,0.0,4.0
2,smart_198_raw,151981829.5,0.000000e+00,0.4094,0.0,0.0
3,smart_9_raw,142648758.0,7.779036e-61,0.3228,27446.0,41391.0
4,smart_4_raw,139317335.5,3.152435e-50,0.2919,10.0,13.0
5,smart_12_raw,134479991.0,1.790119e-36,0.2471,10.0,13.0
6,smart_193_raw,124399750.0,4.915689e-15,0.1536,896.0,1842.0
7,smart_192_raw,121615584.5,6.981984e-11,0.1278,11.0,25.0
8,smart_1_raw,119260271.5,2.040911e-10,0.1059,0.0,0.0
9,smart_7_raw,116650641.0,9.157196e-07,0.0817,0.0,0.0


## 2. Multicollinearity Check: Correlation Matrix

Before treating multiple attributes as if they carry independent signal, check whether
they're just proxies for each other. Spearman (rank) correlation, consistent with the
non-normal reasoning above.

In [3]:
corr = su.correlation_matrix(snapshot, feature_cols)
corr

,smart_1_raw,smart_3_raw,smart_4_raw,smart_5_raw,smart_7_raw,smart_9_raw,smart_10_raw,smart_12_raw,smart_192_raw,smart_193_raw,smart_194_raw,smart_197_raw,smart_198_raw,smart_199_raw
smart_1_raw,1.000000,-0.772228,-0.065655,0.131568,0.940596,0.401869,-0.001417,-0.070755,-0.452985,0.673327,-0.079705,-0.020386,0.006012,0.039097
smart_3_raw,-0.772228,1.000000,0.046204,-0.101142,-0.772779,-0.255557,-0.002193,0.049162,0.067761,-0.759633,-0.027948,0.007532,-0.021435,-0.020029
smart_4_raw,-0.065655,0.046204,1.000000,0.133331,-0.002527,0.180843,-0.002316,0.997974,0.485210,0.138503,0.058583,0.058194,0.057254,0.052596
smart_5_raw,0.131568,-0.101142,0.133331,1.000000,0.161994,0.199490,-0.000430,0.129986,0.023024,0.163977,0.015570,0.311637,0.313210,0.001483
smart_7_raw,0.940596,-0.772779,-0.002527,0.161994,1.000000,0.452412,-0.001415,-0.007470,-0.417673,0.702792,-0.077598,-0.020874,0.009640,0.038234
smart_9_raw,0.401869,-0.255557,0.180843,0.199490,0.452412,1.000000,0.001714,0.172929,-0.160843,0.455977,-0.104844,0.097715,0.108975,0.062240
smart_10_raw,-0.001417,-0.002193,-0.002316,-0.000430,-0.001415,0.001714,1.000000,-0.002290,0.002981,0.000991,0.001472,-0.000276,-0.000230,-0.000235
smart_12_raw,-0.070755,0.049162,0.997974,0.129986,-0.007470,0.172929,-0.002290,1.000000,0.490467,0.135956,0.065368,0.053536,0.052572,0.050892
smart_192_raw,-0.452985,0.067761,0.485210,0.023024,-0.417673,-0.160843,0.002981,0.490467,1.000000,0.079373,0.215729,0.082428,0.072350,-0.031678
smart_193_raw,0.673327,-0.759633,0.138503,0.163977,0.702792,0.455977,0.000991,0.135956,0.079373,1.000000,0.078198,0.051770,0.067405,0.027366


In [4]:
high_corr = su.high_correlation_pairs(corr, threshold=0.7)
print(f"{len(high_corr)} attribute pairs above 0.7 correlation:")
high_corr

7 attribute pairs above 0.7 correlation:


,attribute_1,attribute_2,correlation
0,smart_4_raw,smart_12_raw,0.998
1,smart_1_raw,smart_7_raw,0.941
2,smart_3_raw,smart_7_raw,-0.773
3,smart_1_raw,smart_3_raw,-0.772
4,smart_197_raw,smart_198_raw,0.765
5,smart_3_raw,smart_193_raw,-0.760
6,smart_7_raw,smart_193_raw,0.703


If any pairs show up above 0.7, it may be a signal to pick one
representative attribute per pair rather than treating both as independent evidence. 

## 3. Evaluation: is there a separable signal at all?

A simple logistic regression to check if attributes have a separable signal, 
using ROC-AUC (threshold-independent, appropriate for imbalanced classes) rather
than accuracy, which would be misleadingly high just by predicting "healthy" every time.

In [5]:
result = su.logistic_check(snapshot, feature_cols)

print(f"Train size: {result['n_train']}, Test size: {result['n_test']}")
print(f"ROC-AUC: {result['roc_auc']:.3f}")
print()
print(result['classification_report'])

Train size: 174337, Test size: 74716
ROC-AUC: 0.886

              precision    recall  f1-score   support

           0       1.00      0.92      0.96     74455
           1       0.03      0.75      0.06       261

    accuracy                           0.92     74716
   macro avg       0.52      0.84      0.51     74716
weighted avg       1.00      0.92      0.96     74716



In [6]:
result['coefficients']

,feature,coefficient
0,smart_197_raw,9.094952
1,smart_198_raw,2.173779
2,smart_199_raw,-1.094390
3,smart_5_raw,0.633435
4,smart_9_raw,0.565490
5,smart_7_raw,0.487275
6,smart_3_raw,0.317140
7,smart_12_raw,0.302239
8,smart_193_raw,-0.134186
9,smart_1_raw,-0.123987


**Reading the coefficients:** larger absolute values (after standardization) mean that
attribute contributes more to separating failed from healthy drives *within this simple
model* — this is not the same as effect size, and the two can disagree
when attributes are correlated with each other. Treat this section as a sanity check
that signal exists, not as a ranked "importance" list to quote directly.

**Expectation-setting for the real data:** on this clean synthetic sample, ROC-AUC comes
out very high (near 1.0) because the sample was constructed with an obvious signal built
in. Real SMART data will be much noisier — an AUC anywhere above ~0.65-0.7 using only a
handful of attributes would already be a meaningful, reportable finding.

## 4. Summary: which attributes make the cut?

Combine the univariate test, effect size, and correlation check into one shortlist.
An attribute is a strong candidate for the final write-up if it is:
- statistically significant (p < 0.05), **and**
- has a non-trivial effect size (not just significant because of large sample size), **and**
- isn't a near-duplicate of another attribute already on the list

In [7]:
shortlist = mw_results[mw_results["p_value"] < 0.05].copy()
shortlist = shortlist.reindex(shortlist["effect_size_rank_biserial"].abs().sort_values(ascending=False).index)
shortlist[["attribute", "p_value", "effect_size_rank_biserial", "healthy_median", "failed_median"]]

,attribute,p_value,effect_size_rank_biserial,healthy_median,failed_median
1,smart_197_raw,0.000000e+00,0.6339,0.0,4.0
0,smart_5_raw,0.000000e+00,0.6059,0.0,9.0
2,smart_198_raw,0.000000e+00,0.4094,0.0,0.0
3,smart_9_raw,7.779036e-61,0.3228,27446.0,41391.0
4,smart_4_raw,3.152435e-50,0.2919,10.0,13.0
5,smart_12_raw,1.790119e-36,0.2471,10.0,13.0
6,smart_193_raw,4.915689e-15,0.1536,896.0,1842.0
7,smart_192_raw,6.981984e-11,0.1278,11.0,25.0
8,smart_1_raw,2.040911e-10,0.1059,0.0,0.0
9,smart_7_raw,9.157196e-07,0.0817,0.0,0.0
